# **AFRICAN INSTITUTE FOR MATHEMATICAL SCIENCE - RWANDA**

---
# **Big Data Analytics with Python**
## Assignment 1

### <span style="color:red">Note: all datasets where shared in the classroom</span>
---

#### <span style="color:green">Student Name : Joachim SINYABE DANBE</span>

# **Part 1: Reading, Writing and Validating Data in PySpark HW**

Welcome to your first coding homework assignment in PySpark! I hope you enjoyed the lecture on Reading, Writing and Validating dataframes. Now it's time to put what you've learned into action! 

I've included several instructions below to help guide you through this homework assignment which I hope will get you feeling even comfortable reading, writing and validating dataframes. If you get stuck at any point, feel free to jump to the next lecture where I will guide you through my solutions to the HW assignment. 

Have fun!

Let's dig right in!


## But first things first.....
We need to always begin every Spark session by creating a Spark instance. Let's go ahead and use the method we learned in the lecture in the cell below. Also see if you can remember how to open the Spark UI (using a link that automatically guides you there). 

In [51]:
#import findspark
#findspark.init()

import pyspark
from pyspark.sql import SparkSession 

spark = SparkSession.builder.appName("PGA Tour Data").getOrCreate()
spark

25/11/15 23:18:13 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


## Next let's start by reading a basic csv dataset

Download the pga_tour_historical dataset that is attached to this lecture and save it whatever folder you want, then read it in. 

**Data Source:** https://www.kaggle.com/bradklassen/pga-tour-20102018-data

Rememer to try letting Spark infer the header and infer the Schema types!

In [52]:
path = "Datasets/"
pga = spark.read.csv(path+"pga_tour_historical.csv",inferSchema = True, header = True)


pga.show(5)

print("Schema types : ")
pga.dtypes

+---------------+------+----------------+--------------------+-----+
|    Player Name|Season|       Statistic|            Variable|Value|
+---------------+------+----------------+--------------------+-----+
|Robert Garrigus|  2010|Driving Distance|Driving Distance ...|   71|
|   Bubba Watson|  2010|Driving Distance|Driving Distance ...|   77|
| Dustin Johnson|  2010|Driving Distance|Driving Distance ...|   83|
|Brett Wetterich|  2010|Driving Distance|Driving Distance ...|   54|
|    J.B. Holmes|  2010|Driving Distance|Driving Distance ...|  100|
+---------------+------+----------------+--------------------+-----+
only showing top 5 rows
Schema types : 


[('Player Name', 'string'),
 ('Season', 'int'),
 ('Statistic', 'string'),
 ('Variable', 'string'),
 ('Value', 'string')]

## 1. View first 5 lines of dataframe
First generate a view of the first 5 lines of the dataframe to get an idea of what is inside. We went over two ways of doing this... see if you can remember BOTH ways. 

In [53]:
#1
pga.show(5)

#2
pga.limit(5).toPandas()

+---------------+------+----------------+--------------------+-----+
|    Player Name|Season|       Statistic|            Variable|Value|
+---------------+------+----------------+--------------------+-----+
|Robert Garrigus|  2010|Driving Distance|Driving Distance ...|   71|
|   Bubba Watson|  2010|Driving Distance|Driving Distance ...|   77|
| Dustin Johnson|  2010|Driving Distance|Driving Distance ...|   83|
|Brett Wetterich|  2010|Driving Distance|Driving Distance ...|   54|
|    J.B. Holmes|  2010|Driving Distance|Driving Distance ...|  100|
+---------------+------+----------------+--------------------+-----+
only showing top 5 rows


,Player Name,Season,Statistic,Variable,Value
0,Robert Garrigus,2010,Driving Distance,Driving Distance - (ROUNDS),71
1,Bubba Watson,2010,Driving Distance,Driving Distance - (ROUNDS),77
2,Dustin Johnson,2010,Driving Distance,Driving Distance - (ROUNDS),83
3,Brett Wetterich,2010,Driving Distance,Driving Distance - (ROUNDS),54
4,J.B. Holmes,2010,Driving Distance,Driving Distance - (ROUNDS),100


## 2. Print the schema details

Now print the details of the dataframes schema that Spark infered to ensure that it was infered correctly. Sometimes it is not infered correctly, so we need to watch out!

In [54]:
pga.printSchema()

root
 |-- Player Name: string (nullable = true)
 |-- Season: integer (nullable = true)
 |-- Statistic: string (nullable = true)
 |-- Variable: string (nullable = true)
 |-- Value: string (nullable = true)



## 3. Edit the schema during the read in

We can see from the output above that Spark did not correctly infer that the "value" column was an integer value. Let's try specifying the schema this time to let spark know what the schema should be.

Here is a link to see a list of PySpark data types in case you need it (also attached to the lecture): 
https://spark.apache.org/docs/latest/sql-ref-datatypes.html

In [55]:
from pyspark.sql.types import IntegerType, StructType, StringType, StructField

schema = StructType([
    StructField("Player Name", StringType(), True), 
    StructField("Season", IntegerType(), True),
    StructField("Statistic", StringType(), True),
    StructField("Variable", StringType(), True),
    StructField("Value", IntegerType(), True),
])

pga = spark.read.schema(schema).option("header",True).csv(path+"pga_tour_historical.csv")

pga.printSchema()

root
 |-- Player Name: string (nullable = true)
 |-- Season: integer (nullable = true)
 |-- Statistic: string (nullable = true)
 |-- Variable: string (nullable = true)
 |-- Value: integer (nullable = true)



## 4. Generate summary statistics for only one variable

See if you can generate summary statistics for only the "Value" column using the .describe function

(count, mean, stddev, min, max) 

In [56]:
pga.describe("Value").show()

+-------+------------------+
|summary|             Value|
+-------+------------------+
|  count|           1657247|
|   mean|12494.388998743096|
| stddev|157274.75673570696|
|    min|              -178|
|    max|           3564954|
+-------+------------------+



## 5. Generate summary statistics for TWO variables
Now try to generate ONLY the count min and max for BOTH the "Value" and "Season" variable using the select. You can't use the .describe function for this one but see if you can remember which 
function you CAN use. 


In [57]:
pga.select("Value","Season").summary("count","min","max").show()

+-------+-------+-------+
|summary|  Value| Season|
+-------+-------+-------+
|  count|1657247|2740403|
|    min|   -178|   2010|
|    max|3564954|   2018|
+-------+-------+-------+



## 6. Write a parquet file

Now try writing a parquet file (not partitioned) from the pga dataset. But first create a new dataframe containing ONLY the the "Season" and "Value" fields (using the "select command you used in the question above) and write a parquet file partitioned by "Season". This is a bit of a challenge aimed at getting you ready for material that will be covered later on in the course. Don't feel bad if you can't figure it out.

*Note that if any of your variable names contain spaces, spark will produce an error message with this call. That is why we are selecting ONLY the "Season" and "Value" fields. Ideally we should renamed those columns but we haven't gotten to that yet in this course but we will soon!*

In [58]:
pga_season_value = pga.select("Season","Value")

pga_season_value.show(5)

pga_season_value.write.parquet(path+"Season_not_partitioned", mode = "overwrite"  )

import os
os.listdir(path+'Season_not_partitioned/')

+------+-----+
|Season|Value|
+------+-----+
|  2010|   71|
|  2010|   77|
|  2010|   83|
|  2010|   54|
|  2010|  100|
+------+-----+
only showing top 5 rows


25/11/15 23:18:15 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
25/11/15 23:18:15 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
25/11/15 23:18:15 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 76.00% for 10 writers
25/11/15 23:18:15 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 69.09% for 11 writers
25/11/15 23:18:15 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 63.33% for 12 writers
25/11/15 23:18:15 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 58.46% for 13 writers
25/11/15 23:18:15 WARN MemoryManager: Total allocation exceeds 95.

['part-00007-0115de46-240a-411c-b568-85be6a7fce69-c000.snappy.parquet',
 'part-00003-0115de46-240a-411c-b568-85be6a7fce69-c000.snappy.parquet',
 '.part-00005-0115de46-240a-411c-b568-85be6a7fce69-c000.snappy.parquet.crc',
 'part-00002-0115de46-240a-411c-b568-85be6a7fce69-c000.snappy.parquet',
 'part-00014-0115de46-240a-411c-b568-85be6a7fce69-c000.snappy.parquet',
 'part-00004-0115de46-240a-411c-b568-85be6a7fce69-c000.snappy.parquet',
 '.part-00000-0115de46-240a-411c-b568-85be6a7fce69-c000.snappy.parquet.crc',
 '.part-00015-0115de46-240a-411c-b568-85be6a7fce69-c000.snappy.parquet.crc',
 '.part-00011-0115de46-240a-411c-b568-85be6a7fce69-c000.snappy.parquet.crc',
 '.part-00014-0115de46-240a-411c-b568-85be6a7fce69-c000.snappy.parquet.crc',
 'part-00015-0115de46-240a-411c-b568-85be6a7fce69-c000.snappy.parquet',
 'part-00005-0115de46-240a-411c-b568-85be6a7fce69-c000.snappy.parquet',
 '.part-00007-0115de46-240a-411c-b568-85be6a7fce69-c000.snappy.parquet.crc',
 'part-00010-0115de46-240a-411c-b5

## 7. Write a partioned parquet file

You will need to use the same limited dataframe that you created in the previous question to accomplish this task as well. 

In [59]:
pga_season_value.write.partitionBy("Season").parquet(path+"Season_partitioned", mode = "overwrite"  )


import os
os.listdir(path+'Season_partitioned/')

['Season=2015',
 'Season=2017',
 'Season=2012',
 'Season=2016',
 'Season=2013',
 '_SUCCESS',
 'Season=2014',
 'Season=2010',
 'Season=2011',
 '._SUCCESS.crc',
 'Season=2018']

## 8. Read in a partitioned parquet file

Now try reading in the partitioned parquet file you just created above. 

In [60]:
season_partitioned = spark.read.parquet(path+"Season_partitioned")

season_partitioned.show(4)

season_partitioned.count()

+-----+------+
|Value|Season|
+-----+------+
|   71|  2010|
|   77|  2010|
|   83|  2010|
|   54|  2010|
+-----+------+
only showing top 4 rows


2740403

## 9. Reading in a set of paritioned parquet files

Now try only reading Seasons 2010, 2011 and 2012.

In [61]:
season_2010_2011_2012 = season_partitioned.filter(season_partitioned["Season"].isin([2010, 20111, 2012]))
season_2010_2011_2012.show(5)


# x_last = season_2010_2011_2012.toPandas()
# x_last.tail(8)

+-----+------+
|Value|Season|
+-----+------+
|   71|  2010|
|   77|  2010|
|   83|  2010|
|   54|  2010|
|  100|  2010|
+-----+------+
only showing top 5 rows


## 10. Create your own dataframe

Try creating your own dataframe below using PySparks *.createDataFrame* function. See if you can make one that contains 4 variables and at least 3 rows. 

Let's see how creative you can get on the content of the dataframe :)

In [62]:
from pyspark.sql import Row

data = [
    Row(Name= "Joachim", Major="Python Programming", Grade=94,Year=2025),  
    Row(Name="Julie", Major="Physics", Grade=91, Year=2022),
    Row(Name="Alex", Major="Math", Grade=85, Year=2022),
    Row(Name="Sam", Major="Computer Science", Grade=78, Year=2023)
]
students = spark.createDataFrame(data)

print("==============List of students ===================")
students.show()

print("==============Schema of students ===================")
students.printSchema()


students.filter(students.Grade > 80).show()
students.groupBy("Year").avg("Grade").show()


==============List of students ===================
+-------+------------------+-----+----+
|   Name|             Major|Grade|Year|
+-------+------------------+-----+----+
|Joachim|Python Programming|   94|2025|
|  Julie|           Physics|   91|2022|
|   Alex|              Math|   85|2022|
|    Sam|  Computer Science|   78|2023|
+-------+------------------+-----+----+

==============Schema of students ===================
root
 |-- Name: string (nullable = true)
 |-- Major: string (nullable = true)
 |-- Grade: long (nullable = true)
 |-- Year: long (nullable = true)

+-------+------------------+-----+----+
|   Name|             Major|Grade|Year|
+-------+------------------+-----+----+
|Joachim|Python Programming|   94|2025|
|  Julie|           Physics|   91|2022|
|   Alex|              Math|   85|2022|
+-------+------------------+-----+----+

+----+----------+
|Year|avg(Grade)|
+----+----------+
|2025|      94.0|
|2022|      88.0|
|2023|      78.0|
+----+----------+



# **Part 2: Manipulating Data in DataFrames HW**


#### Let's get started applying what we learned in the lecure!

I've provided several questions below to help test and expand you knowledge from the code along lecture. So let's see what you've got!

First create your spark instance as we need to do at the start of every project.

In [63]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("PGA Manipulation").getOrCreate()

spark

25/11/15 23:18:18 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


## Read in our Republican vs. Democrats Tweet DataFrame

Attached to the lecture

In [64]:
path = "Datasets/"
df_Rep_vs_Demo  = spark.read.csv(path+"ExtractedTweets.csv", inferSchema = True, header = True )


## About this dataframe

Extracted tweets from all of the representatives (latest 200 as of May 17th 2018)

**Source:** https://www.kaggle.com/kapastor/democratvsrepublicantweets#ExtractedTweets.csv

Use either .show() or .toPandas() check out the first view rows of the dataframe to get an idea of what we are working with.

In [65]:
df_Rep_vs_Demo.show(5)
df_Rep_vs_Demo.limit(5).toPandas()

+--------------------+-------------+--------------------+
|               Party|       Handle|               Tweet|
+--------------------+-------------+--------------------+
|            Democrat|RepDarrenSoto|Today, Senate Dem...|
|            Democrat|RepDarrenSoto|RT @WinterHavenSu...|
|            Democrat|RepDarrenSoto|RT @NBCLatino: .@...|
|Congress has allo...|         NULL|                NULL|
|            Democrat|RepDarrenSoto|RT @NALCABPolicy:...|
+--------------------+-------------+--------------------+
only showing top 5 rows


,Party,Handle,Tweet
0,Democrat,RepDarrenSoto,"Today, Senate Dems vote to #SaveTheInternet. P..."
1,Democrat,RepDarrenSoto,RT @WinterHavenSun: Winter Haven resident / Al...
2,Democrat,RepDarrenSoto,RT @NBCLatino: .@RepDarrenSoto noted that Hurr...
3,"Congress has allocated about $18…""",None,None
4,Democrat,RepDarrenSoto,RT @NALCABPolicy: Meeting with @RepDarrenSoto ...


**Prevent Truncation of view**

If the view you produced above truncated some of the longer tweets, see if you can prevent that so you can read the whole tweet.

In [66]:
df_Rep_vs_Demo.show(5, truncate = False)

+----------------------------------+-------------+--------------------------------------------------------------------------------------------------------------------------------------------+
|Party                             |Handle       |Tweet                                                                                                                                       |
+----------------------------------+-------------+--------------------------------------------------------------------------------------------------------------------------------------------+
|Democrat                          |RepDarrenSoto|Today, Senate Dems vote to #SaveTheInternet. Proud to support similar #NetNeutrality legislation here in the House… https://t.co/n3tggDLU1L |
|Democrat                          |RepDarrenSoto|RT @WinterHavenSun: Winter Haven resident / Alta Vista teacher is one of several recognized by @RepDarrenSoto for National Teacher Apprecia…|
|Democrat                          |RepD

**Print Schema**

First, check the schema to make sure the datatypes are accurate. 

In [67]:
df_Rep_vs_Demo.printSchema()

root
 |-- Party: string (nullable = true)
 |-- Handle: string (nullable = true)
 |-- Tweet: string (nullable = true)



## 1. Can you identify any tweet that mentions the handle @LatinoLeader using regexp_extract?

It doesn't matter how you identify the row, any identifier will do. You can test your script on row 5 from this dataset. That row contains @LatinoLeader. 

In [68]:
from pyspark.sql.functions import regexp_extract

handle_pattern = r"(@LatinoLeader)"


tweets_handle = df_Rep_vs_Demo.withColumn(
    "founds_handle",
    regexp_extract("Tweet", handle_pattern, 1)
)

tweets_latino = tweets_handle.filter(tweets_handle["founds_handle"] != "")
tweets_latino.limit(6).toPandas()

,Party,Handle,Tweet,founds_handle
0,Democrat,RepDarrenSoto,RT @NALCABPolicy: Meeting with @RepDarrenSoto ...,@LatinoLeader


## 2. Replace any value other than 'Democrate' or 'Republican' with 'Other' in the Party column.

We can see from the output below, that there are several other values other than 'Democrate' or 'Republican' in the Part column. We are assuming that this is dirty data that needs to be cleaned up.

In [69]:
from pyspark.sql.functions import when

tweets_cleaned = df_Rep_vs_Demo.withColumn(
    "Party",
    when(
        (df_Rep_vs_Demo["Party"] == "Democrat") | (df_Rep_vs_Demo["Party"] == "Republican"),
        df_Rep_vs_Demo["Party"]
    ).otherwise("Other")
)

tweets_cleaned.select("Party").distinct().show(3)

+----------+
|     Party|
+----------+
|  Democrat|
|     Other|
|Republican|
+----------+



## 3. Delete all embedded links (ie. "https:....)

For example see the first row in the tweets dataframe. 

*Note: this may require an google search :)*

In [70]:
from pyspark.sql.functions import regexp_replace

tweets_no_links = tweets_cleaned.withColumn(
    "Tweet",
    regexp_replace("Tweet", r"http[s]?://\S+","")
)

tweets_no_links.show(5, False)

+--------+-------------+--------------------------------------------------------------------------------------------------------------------------------------------+
|Party   |Handle       |Tweet                                                                                                                                       |
+--------+-------------+--------------------------------------------------------------------------------------------------------------------------------------------+
|Democrat|RepDarrenSoto|Today, Senate Dems vote to #SaveTheInternet. Proud to support similar #NetNeutrality legislation here in the House…                         |
|Democrat|RepDarrenSoto|RT @WinterHavenSun: Winter Haven resident / Alta Vista teacher is one of several recognized by @RepDarrenSoto for National Teacher Apprecia…|
|Democrat|RepDarrenSoto|RT @NBCLatino: .@RepDarrenSoto noted that Hurricane Maria has left approximately $90 billion in damages.                                    |
|Oth

## 4. Remove any leading or trailing white space in the tweet column

In [71]:
from pyspark.sql.functions import trim

tweets_clean = tweets_no_links.withColumn(
    "Tweet",
    trim("Tweet")
)

tweets_clean.show()

+--------+-------------+--------------------+
|   Party|       Handle|               Tweet|
+--------+-------------+--------------------+
|Democrat|RepDarrenSoto|Today, Senate Dem...|
|Democrat|RepDarrenSoto|RT @WinterHavenSu...|
|Democrat|RepDarrenSoto|RT @NBCLatino: .@...|
|   Other|         NULL|                NULL|
|Democrat|RepDarrenSoto|RT @NALCABPolicy:...|
|Democrat|RepDarrenSoto|RT @Vegalteno: Hu...|
|Democrat|RepDarrenSoto|RT @EmgageActionF...|
|Democrat|RepDarrenSoto|Hurricane Maria l...|
|Democrat|RepDarrenSoto|RT @Tharryry: I a...|
|Democrat|RepDarrenSoto|RT @HispanicCaucu...|
|Democrat|RepDarrenSoto|RT @RepStephMurph...|
|Democrat|RepDarrenSoto|RT @AllSaints_FL:...|
|Democrat|RepDarrenSoto|.@realDonaldTrump...|
|Democrat|RepDarrenSoto|Thank you to my m...|
|Democrat|RepDarrenSoto|We paid our respe...|
|   Other|         NULL|                NULL|
|Democrat|RepDarrenSoto|RT @WinterHavenSu...|
|Democrat|RepDarrenSoto|Meet 12 incredibl...|
|Democrat|RepDarrenSoto|RT @wildli

## 5. Rename the 'Party' column to 'Dem_Rep'

No real reason here :) just wanted you to get practice doing this. 

In [72]:
tweets_renamed  = tweets_clean.withColumnRenamed("Party","Dem_Rep")

tweets_renamed.printSchema()


root
 |-- Dem_Rep: string (nullable = true)
 |-- Handle: string (nullable = true)
 |-- Tweet: string (nullable = true)



## 6. Concatenate the Party and Handle columns

Silly yes... but good practice.

pyspark.sql.functions.concat_ws(sep, *cols)[source] <br>
Concatenates multiple input string columns together into a single string column, using the given separator.

In [73]:
from pyspark.sql.functions import concat_ws

tweets_concat = tweets_clean.withColumn(
    "Party_Handle",
    concat_ws("_", tweets_clean["Party"], tweets_clean["Handle"])
)
tweets_concat.show(5)


+--------+-------------+--------------------+--------------------+
|   Party|       Handle|               Tweet|        Party_Handle|
+--------+-------------+--------------------+--------------------+
|Democrat|RepDarrenSoto|Today, Senate Dem...|Democrat_RepDarre...|
|Democrat|RepDarrenSoto|RT @WinterHavenSu...|Democrat_RepDarre...|
|Democrat|RepDarrenSoto|RT @NBCLatino: .@...|Democrat_RepDarre...|
|   Other|         NULL|                NULL|               Other|
|Democrat|RepDarrenSoto|RT @NALCABPolicy:...|Democrat_RepDarre...|
+--------+-------------+--------------------+--------------------+
only showing top 5 rows


## Challenge Question

Let's image that we want to analyze the hashtags that are used in these tweets. Can you extract all the hashtags you see?

In [74]:
from pyspark.sql.functions import expr

tweets_hashtags = tweets_concat.withColumn(
    "Hashtags", 
    expr(r"regexp_extract_all(Tweet, r'(#\w+)',0)")
)
tweets_hashtags.show(5, False)
tweets_hashtags.printSchema()

+--------+-------------+--------------------------------------------------------------------------------------------------------------------------------------------+----------------------+----------------------------------+
|Party   |Handle       |Tweet                                                                                                                                       |Party_Handle          |Hashtags                          |
+--------+-------------+--------------------------------------------------------------------------------------------------------------------------------------------+----------------------+----------------------------------+
|Democrat|RepDarrenSoto|Today, Senate Dems vote to #SaveTheInternet. Proud to support similar #NetNeutrality legislation here in the House…                         |Democrat_RepDarrenSoto|[#SaveTheInternet, #NetNeutrality]|
|Democrat|RepDarrenSoto|RT @WinterHavenSun: Winter Haven resident / Alta Vista teacher is one of several

# Let's create our own dataset to work with real dates

This is a dataset of patient visits from a medical office. It contains the patients first and last names, date of birth, and the dates of their first 3 visits. 

In [75]:
from pyspark.sql.types import *

md_office = [('Mohammed','Alfasy','1987-4-8','2016-1-7','2017-2-3','2018-3-2') \
            ,('Marcy','Wellmaker','1986-4-8','2015-1-7','2017-1-3','2018-1-2') \
            ,('Ginny','Ginger','1986-7-10','2014-8-7','2015-2-3','2016-3-2') \
            ,('Vijay','Doberson','1988-5-2','2016-1-7','2018-2-3','2018-3-2') \
            ,('Orhan','Gelicek','1987-5-11','2016-5-7','2017-1-3','2018-9-2') \
            ,('Sarah','Jones','1956-7-6','2016-4-7','2017-8-3','2018-10-2') \
            ,('John','Johnson','2017-10-12','2018-1-2','2018-10-3','2018-3-2') ]

df = spark.createDataFrame(md_office,['first_name','last_name','dob','visit1','visit2','visit3']) # schema=final_struc

# Check to make sure it worked
df.show()
print(df.printSchema())

+----------+---------+----------+--------+---------+---------+
|first_name|last_name|       dob|  visit1|   visit2|   visit3|
+----------+---------+----------+--------+---------+---------+
|  Mohammed|   Alfasy|  1987-4-8|2016-1-7| 2017-2-3| 2018-3-2|
|     Marcy|Wellmaker|  1986-4-8|2015-1-7| 2017-1-3| 2018-1-2|
|     Ginny|   Ginger| 1986-7-10|2014-8-7| 2015-2-3| 2016-3-2|
|     Vijay| Doberson|  1988-5-2|2016-1-7| 2018-2-3| 2018-3-2|
|     Orhan|  Gelicek| 1987-5-11|2016-5-7| 2017-1-3| 2018-9-2|
|     Sarah|    Jones|  1956-7-6|2016-4-7| 2017-8-3|2018-10-2|
|      John|  Johnson|2017-10-12|2018-1-2|2018-10-3| 2018-3-2|
+----------+---------+----------+--------+---------+---------+

root
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- dob: string (nullable = true)
 |-- visit1: string (nullable = true)
 |-- visit2: string (nullable = true)
 |-- visit3: string (nullable = true)

None


Oh no! The dates are still stored as text... let's try converting them again and see if we have any issues this time.

In [76]:
from pyspark.sql.functions import to_date

date_format = "yyyy-M-d"

df_converted = df.withColumn("dob", to_date("dob", date_format)) \
    .withColumn("visit1", to_date("visit1", date_format)) \
    .withColumn("visit2", to_date("visit2", date_format)) \
    .withColumn("visit3", to_date("visit3", date_format))


df_converted.show(truncate=False)
df_converted.printSchema()


+----------+---------+----------+----------+----------+----------+
|first_name|last_name|dob       |visit1    |visit2    |visit3    |
+----------+---------+----------+----------+----------+----------+
|Mohammed  |Alfasy   |1987-04-08|2016-01-07|2017-02-03|2018-03-02|
|Marcy     |Wellmaker|1986-04-08|2015-01-07|2017-01-03|2018-01-02|
|Ginny     |Ginger   |1986-07-10|2014-08-07|2015-02-03|2016-03-02|
|Vijay     |Doberson |1988-05-02|2016-01-07|2018-02-03|2018-03-02|
|Orhan     |Gelicek  |1987-05-11|2016-05-07|2017-01-03|2018-09-02|
|Sarah     |Jones    |1956-07-06|2016-04-07|2017-08-03|2018-10-02|
|John      |Johnson  |2017-10-12|2018-01-02|2018-10-03|2018-03-02|
+----------+---------+----------+----------+----------+----------+

root
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- dob: date (nullable = true)
 |-- visit1: date (nullable = true)
 |-- visit2: date (nullable = true)
 |-- visit3: date (nullable = true)



## 7. Can you calculate a variable showing the length of time between patient visits?

Compare visit1 to visit2 and visit2 to visit3 for all patients and see what the average length of time is between visits. Create an alias for it as well. 

In [77]:
from pyspark.sql.functions import datediff, mean

df_with_diffs = df_converted.withColumn(
    "days_btwn_v1_v2", datediff("visit2", "visit1")
).withColumn(
    "days_btwn_v2_v3", datediff("visit3", "visit2")
)

df_with_diffs.show()

averages = df_with_diffs.agg(
    mean("days_btwn_v1_v2").alias("avg_days_btwn_v1_v2"),
    mean("days_btwn_v2_v3").alias("avg_days_btwn_v2_v3")
)
averages.show()


df_with_avgs = df_with_diffs.withColumn(
    "mean_days_btwn",
    expr("(days_btwn_v1_v2 + days_btwn_v2_v3)/2")
)
df_with_avgs.select("first_name", "last_name", "mean_days_btwn").show()



+----------+---------+----------+----------+----------+----------+---------------+---------------+
|first_name|last_name|       dob|    visit1|    visit2|    visit3|days_btwn_v1_v2|days_btwn_v2_v3|
+----------+---------+----------+----------+----------+----------+---------------+---------------+
|  Mohammed|   Alfasy|1987-04-08|2016-01-07|2017-02-03|2018-03-02|            393|            392|
|     Marcy|Wellmaker|1986-04-08|2015-01-07|2017-01-03|2018-01-02|            727|            364|
|     Ginny|   Ginger|1986-07-10|2014-08-07|2015-02-03|2016-03-02|            180|            393|
|     Vijay| Doberson|1988-05-02|2016-01-07|2018-02-03|2018-03-02|            758|             27|
|     Orhan|  Gelicek|1987-05-11|2016-05-07|2017-01-03|2018-09-02|            241|            607|
|     Sarah|    Jones|1956-07-06|2016-04-07|2017-08-03|2018-10-02|            483|            425|
|      John|  Johnson|2017-10-12|2018-01-02|2018-10-03|2018-03-02|            274|           -215|
+---------

## 8. Can you calculate the age of each patient?

In [78]:
from pyspark.sql.functions import datediff, current_date

df_with_age = df_converted.withColumn(
    "age",
    (datediff(current_date(), "dob") / 365).cast("integer")
)

df_with_age.select("first_name", "last_name", "dob", "age").show()


+----------+---------+----------+---+
|first_name|last_name|       dob|age|
+----------+---------+----------+---+
|  Mohammed|   Alfasy|1987-04-08| 38|
|     Marcy|Wellmaker|1986-04-08| 39|
|     Ginny|   Ginger|1986-07-10| 39|
|     Vijay| Doberson|1988-05-02| 37|
|     Orhan|  Gelicek|1987-05-11| 38|
|     Sarah|    Jones|1956-07-06| 69|
|      John|  Johnson|2017-10-12|  8|
+----------+---------+----------+---+



## 9. Can you extract the month from the first visit column and call it "Month"?

In [79]:
from pyspark.sql.functions import month

df_with_month = df_with_age.withColumn(
    "Month",
    month("visit1")
)

df_with_month.select("first_name", "last_name", "visit1", "Month").show()


+----------+---------+----------+-----+
|first_name|last_name|    visit1|Month|
+----------+---------+----------+-----+
|  Mohammed|   Alfasy|2016-01-07|    1|
|     Marcy|Wellmaker|2015-01-07|    1|
|     Ginny|   Ginger|2014-08-07|    8|
|     Vijay| Doberson|2016-01-07|    1|
|     Orhan|  Gelicek|2016-05-07|    5|
|     Sarah|    Jones|2016-04-07|    4|
|      John|  Johnson|2018-01-02|    1|
+----------+---------+----------+-----+



## 10. Challenges with working with date and timestamps

Let's read in the supermarket sales dataframe attached to the le
cture now and see some of the issues that can come up when working with date and timestamps values.

In [80]:
sales_path = "Datasets/supermarket_sales.csv"

df_sales = spark.read.csv(sales_path,header=True,inferSchema=False)

## About this dataset

The growth of supermarkets in most populated cities are increasing and market competitions are also high. The dataset is one of the historical sales of supermarket company which has recorded in 3 different branches for 3 months data. 

 - Attribute information
 - Invoice id: Computer generated sales slip invoice identification number
 - Branch: Branch of supercenter (3 branches are available identified by A, B and C).
 - City: Location of supercenters
 - Customer type: Type of customers, recorded by Members for customers using member card and Normal for without member card.
 - Gender: Gender type of customer
 - Product line: General item categorization groups - Electronic accessories, Fashion accessories, Food and beverages, Health and beauty, Home and lifestyle, Sports and travel
 - Unit price: Price of each product in USD
 - Quantity: Number of products purchased by customer
 - Tax: 5% tax fee for customer buying
 - Total: Total price including tax
 - Date: Date of purchase (Record available from January 2019 to March 2019)
 - Time: Purchase time (10am to 9pm)
 - Payment: Payment used by customer for purchase (3 methods are available – Cash, Credit card and Ewallet)
 - COGS: Cost of goods sold
 - Gross margin percentage: Gross margin percentage
 - Gross income: Gross income
 - Rating: Customer stratification rating on their overall shopping experience (On a scale of 1 to 10)

**Source:** https://www.kaggle.com/aungpyaeap/supermarket-sales

### View dataframe and schema as usual

In [81]:
df_sales.show(5)

df_sales.printSchema()

df_sales.limit(4).toPandas()

+-----------+------+---------+-------------+------+--------------------+----------+--------+-------+--------+---------+-----+-----------+------+-----------------------+------------+------+
| Invoice ID|Branch|     City|Customer type|Gender|        Product line|Unit price|Quantity| Tax 5%|   Total|     Date| Time|    Payment|  cogs|gross margin percentage|gross income|Rating|
+-----------+------+---------+-------------+------+--------------------+----------+--------+-------+--------+---------+-----+-----------+------+-----------------------+------------+------+
|750-67-8428|     A|   Yangon|       Member|Female|   Health and beauty|     74.69|       7|26.1415|548.9715| 1/5/2019|13:08|    Ewallet|522.83|            4.761904762|     26.1415|   9.1|
|226-31-3081|     C|Naypyitaw|       Normal|Female|Electronic access...|     15.28|       5|   3.82|   80.22| 3/8/2019|10:29|       Cash|  76.4|            4.761904762|        3.82|   9.6|
|631-41-3108|     A|   Yangon|       Normal|  Male|  Ho

,Invoice ID,Branch,City,Customer type,Gender,Product line,Unit price,Quantity,Tax 5%,Total,Date,Time,Payment,cogs,gross margin percentage,gross income,Rating
0,750-67-8428,A,Yangon,Member,Female,Health and beauty,74.69,7,26.1415,548.9715,1/5/2019,13:08,Ewallet,522.83,4.761904762,26.1415,9.1
1,226-31-3081,C,Naypyitaw,Normal,Female,Electronic accessories,15.28,5,3.82,80.22,3/8/2019,10:29,Cash,76.4,4.761904762,3.82,9.6
2,631-41-3108,A,Yangon,Normal,Male,Home and lifestyle,46.33,7,16.2155,340.5255,3/3/2019,13:23,Credit card,324.31,4.761904762,16.2155,7.4
3,123-19-1176,A,Yangon,Member,Male,Health and beauty,58.22,8,23.288,489.048,1/27/2019,20:33,Ewallet,465.76,4.761904762,23.288,8.4


### Convert date field to date type

Looks like we need to convert the date field into a date type. Let's go ahead and do that..

In [82]:
from pyspark.sql.functions import to_date

df_sales = df_sales.withColumn("Date", to_date("Date", "M/d/yyyy"))
df_sales.select("Date").show(3, False)


df_sales.printSchema()

+----------+
|Date      |
+----------+
|2019-01-05|
|2019-03-08|
|2019-03-03|
+----------+
only showing top 3 rows
root
 |-- Invoice ID: string (nullable = true)
 |-- Branch: string (nullable = true)
 |-- City: string (nullable = true)
 |-- Customer type: string (nullable = true)
 |-- Gender: string (nullable = true)
 |-- Product line: string (nullable = true)
 |-- Unit price: string (nullable = true)
 |-- Quantity: string (nullable = true)
 |-- Tax 5%: string (nullable = true)
 |-- Total: string (nullable = true)
 |-- Date: date (nullable = true)
 |-- Time: string (nullable = true)
 |-- Payment: string (nullable = true)
 |-- cogs: string (nullable = true)
 |-- gross margin percentage: string (nullable = true)
 |-- gross income: string (nullable = true)
 |-- Rating: string (nullable = true)



### How can we extract the month value from the date field?

If you had trouble converting the date field in the previous question think about a more creative solution to extract the month from that field.

In [83]:
from pyspark.sql.functions import month, to_date

# Suppose your date format is "yyyy-MM-dd"
df_sales = df_sales.withColumn("Month", month(to_date("Date", "yyyy-MM-dd")))

df_sales.select("Date", "Month").show(10, False)



+----------+-----+
|Date      |Month|
+----------+-----+
|2019-01-05|1    |
|2019-03-08|3    |
|2019-03-03|3    |
|2019-01-27|1    |
|2019-02-08|2    |
|2019-03-25|3    |
|2019-02-25|2    |
|2019-02-24|2    |
|2019-01-10|1    |
|2019-02-20|2    |
+----------+-----+
only showing top 10 rows


## 11.0 Working with Arrays

Here is a dataframe of reviews from the movie the Dark Night.

In [84]:
from pyspark.sql.functions import *

values = [(5,'Epic. This is the best movie I have EVER seen'), \
          (4,'Pretty good, but I would have liked to seen better special effects'), \
          (3,'So so. Casting could have been improved'), \
          (5,'The most EPIC movie of the year! Casting was awesome. Special effects were so intense.'), \
          (4,'Solid but I would have liked to see more of the love story'), \
          (5,'THE BOMB!!!!!!!')]
reviews = spark.createDataFrame(values,['rating', 'review_txt'])

reviews.show(6,False)

+------+--------------------------------------------------------------------------------------+
|rating|review_txt                                                                            |
+------+--------------------------------------------------------------------------------------+
|5     |Epic. This is the best movie I have EVER seen                                         |
|4     |Pretty good, but I would have liked to seen better special effects                    |
|3     |So so. Casting could have been improved                                               |
|5     |The most EPIC movie of the year! Casting was awesome. Special effects were so intense.|
|4     |Solid but I would have liked to see more of the love story                            |
|5     |THE BOMB!!!!!!!                                                                       |
+------+--------------------------------------------------------------------------------------+



## 11.1 Let's see if we can create an array off of the review text column and then derive some meaningful results from it.

**But first** we need to clean the rview_txt column to make sure we can get what we need from our analysis later on. So let's do the following:

1. Remove all punctuation
2. lower case everything
3. Remove white space (trim)
3. Then finally, split the string

In [85]:
reviews_clean = (
    reviews
    .withColumn("clean_txt",
        trim(lower(regexp_replace(col("review_txt"), "[^A-Za-z0-9 ]", "")))
    )
    .withColumn("word_array", split(col("clean_txt"), " "))
)

reviews_clean.show(6,False)


+------+--------------------------------------------------------------------------------------+-----------------------------------------------------------------------------------+---------------------------------------------------------------------------------------------------+
|rating|review_txt                                                                            |clean_txt                                                                          |word_array                                                                                         |
+------+--------------------------------------------------------------------------------------+-----------------------------------------------------------------------------------+---------------------------------------------------------------------------------------------------+
|5     |Epic. This is the best movie I have EVER seen                                         |epic this is the best movie i have ever seen                     

## 11.2 Alright now let's see if we can find which reviews contain the word 'Epic'

In [86]:
epic_reviews = reviews_clean.filter(array_contains(col("word_array"), "epic"))

epic_reviews.show(truncate=False)

+------+--------------------------------------------------------------------------------------+-----------------------------------------------------------------------------------+---------------------------------------------------------------------------------------------------+
|rating|review_txt                                                                            |clean_txt                                                                          |word_array                                                                                         |
+------+--------------------------------------------------------------------------------------+-----------------------------------------------------------------------------------+---------------------------------------------------------------------------------------------------+
|5     |Epic. This is the best movie I have EVER seen                                         |epic this is the best movie i have ever seen                     

# **Part 3: Joining and Appending DataFrames in PySpark HW**

Now it's time to test your knowledge and further engrain the concepts we touched on in the lectures. Let's go ahead and get started.




**As always let's start our Spark instance.**

In [87]:
spark = SparkSession.builder.appName("UWMadisonCourses").getOrCreate()
spark

25/11/15 23:18:21 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


## Read in the database

Let cotinue working with our college courses dataframe to get some more insights and practice what we have learned!Let's read in the whole database using the loop function that we learned about in the lecture to automatically read in all the datasets from the uw-madision-courses folder (there are too many datasets to each one individually.

In [88]:
import zipfile
import os

zip_path = "Datasets/archive.zip"
path3 = "Datasets/archive" 
os.makedirs(path3, exist_ok=True)



with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(path3)

dataframes = {}

dataframes = {}
for file in os.listdir(path3):
     if file.endswith(".csv"):
         table_name = file.replace(".csv", "")
         df = spark.read.csv(os.path.join(path3, file), header=True, inferSchema=True)
         dataframes[table_name] = df


Now check the contents of a few of the dataframses that were read in above.

In [89]:
dataframes['course_offerings'].show(5)
dataframes['instructors'].show(5)
dataframes['sections'].show(5)

+--------------------+--------------------+---------+--------------------+
|                uuid|         course_uuid|term_code|                name|
+--------------------+--------------------+---------+--------------------+
|344b3ebe-da7e-314...|a3e3e1c3-543d-3bb...|     1092|Cooperative Educa...|
|f718e6cd-33f0-3c1...|a3e3e1c3-543d-3bb...|     1082|Cooperative Educa...|
|ea3b717c-d66b-30d...|a3e3e1c3-543d-3bb...|     1172|Cooperative Educa...|
|075da420-5f49-3dd...|a3e3e1c3-543d-3bb...|     1114|Cooperative Educa...|
|2b4e216d-a728-371...|a3e3e1c3-543d-3bb...|     1104|Cooperative Educa...|
+--------------------+--------------------+---------+--------------------+
only showing top 5 rows
+-------+------------------+
|     id|              name|
+-------+------------------+
| 761703|  JOHN ARCHAMBAULT|
|3677061|    STEPHANIE KANN|
| 788586|        KATHY PREM|
|1600463|KRISTIN KLARKOWSKI|
| 693634|    DAVID BOHNHOFF|
+-------+------------------+
only showing top 5 rows
+---------------

## Recap: About this database

You will notice that there are several more tables in the uw-madision-courses folder than there are read in above. This so that you will have a chance to practice your own custom joins and learn about the relationships between a real database work. Sometimes we don't know how they are related and we need to figure it out! I'll save that for the HW :) 

Here is a look at some of the important variables we can use to join our tables:

 - course_offerings: uuid, course_uuid, term_code, name
 - instructors: id, name
 - schedules: uuid
 - sections: uuid, course_offering_uuid,room_uuid, schedule_uuid
 - teachings: instructor_id, section_uuid
 - courses: uuid
 - grade_distributions: course_offering_uuid,section_number
 - rooms: uuid, facility_code, room_code
 - subjects: code
 - subject_memberships: subject_code, course_offering_uuid
 
 **Source:** https://www.kaggle.com/Madgrades/uw-madison-courses
 
So alright, let's use this information to discover some insights from this data!

## 1a. Can you assign the room numbers to each section of each course?

Show only the rooms uuid, facility code, room number, term code and the name of the course from the course_offerings table.

In [90]:
from pyspark.sql.functions import col

sections_rooms = dataframes['sections'].join(
    dataframes['rooms'].alias('r'),
    dataframes['sections']['room_uuid'] == col('r.uuid'),
    'left'
)
sections_rooms_offerings = sections_rooms.join(
    dataframes['course_offerings'].alias('c'),
    sections_rooms['course_offering_uuid'] == col('c.uuid'),
    'left'
)

columns_aliased = [
    col('r.uuid').alias('room_uuid'),       
    col('r.facility_code'),
    col('r.room_code'),
    col('c.term_code'),
    col('c.name')
]
sections_rooms_offerings.select(*columns_aliased).show(10)
sections_rooms_offerings.printSchema()

+--------------------+-------------+---------+---------+--------------------+
|           room_uuid|facility_code|room_code|term_code|                name|
+--------------------+-------------+---------+---------+--------------------+
|                NULL|         NULL|     NULL|     1092|Cooperative Educa...|
|                NULL|         NULL|     NULL|     1082|Cooperative Educa...|
|04368a56-c959-3e4...|   OFF CAMPUS|     null|     1172|Cooperative Educa...|
|                NULL|         NULL|     NULL|     1172|Cooperative Educa...|
|                NULL|         NULL|     NULL|     1172|Cooperative Educa...|
|04368a56-c959-3e4...|   OFF CAMPUS|     null|     1172|Cooperative Educa...|
|04368a56-c959-3e4...|   OFF CAMPUS|     null|     1114|Cooperative Educa...|
|                NULL|         NULL|     NULL|     1114|Cooperative Educa...|
|                NULL|         NULL|     NULL|     1104|Cooperative Educa...|
|04368a56-c959-3e4...|   OFF CAMPUS|     null|     1104|Cooperat

## 1b. Now show same output as above but for only facility number 0469 (facility_code)

In [91]:
resultb = sections_rooms_offerings.filter(col('facility_code') == '0469')
resultb.select(columns_aliased).show(10)

+--------------------+-------------+---------+---------+------------------+
|           room_uuid|facility_code|room_code|term_code|              name|
+--------------------+-------------+---------+---------+------------------+
|9759cb5f-a7d3-3d0...|         0469|     2441|     1152|Fundamentals-Flute|
|9759cb5f-a7d3-3d0...|         0469|     2441|     1092|Fundamentals-Flute|
|6af80b0b-b3e3-370...|         0469|     4411|     1172|Fundamentals-Flute|
|9759cb5f-a7d3-3d0...|         0469|     2441|     1162|Fundamentals-Flute|
|9759cb5f-a7d3-3d0...|         0469|     2441|     1132|Fundamentals-Flute|
|50322d30-dd8f-3c6...|         0469|     2511|     1072|Fundamentals-Flute|
|9759cb5f-a7d3-3d0...|         0469|     2441|     1142|Fundamentals-Flute|
|9759cb5f-a7d3-3d0...|         0469|     2441|     1112|Fundamentals-Flute|
|9759cb5f-a7d3-3d0...|         0469|     2441|     1102|Fundamentals-Flute|
|50322d30-dd8f-3c6...|         0469|     2511|     1082|Fundamentals-Flute|
+-----------

## 2. Count how many sections are offered for each subject for each facility

*Note: this will involve a groupby*

In [92]:
subs_co = dataframes['subject_memberships'].join(
    dataframes['course_offerings'],
    dataframes['subject_memberships']['course_offering_uuid'] == dataframes['course_offerings']['uuid'],
    'left'
)
sections_join = dataframes['sections'] \
    .join(subs_co, dataframes['sections']['course_offering_uuid'] == subs_co['uuid'], 'left') \
    .join(dataframes['rooms'], dataframes['sections']['room_uuid'] == dataframes['rooms']['uuid'], 'left')

from pyspark.sql import functions as F
section_counts = sections_join.groupBy(
    'subject_code', 'facility_code'
).agg(
    F.count('*').alias('section_count')
)
section_counts.orderBy('subject_code', 'facility_code').show(20)


+------------+-------------+-------------+
|subject_code|facility_code|section_count|
+------------+-------------+-------------+
|         102|         NULL|           16|
|         102|         0025|           42|
|         102|         0503|           30|
|         102|         1095|          211|
|         104|         NULL|          197|
|         104|         0000|            1|
|         104|         0018|           11|
|         104|         0046|          182|
|         104|         0047|           13|
|         104|         0048|           30|
|         104|         0050|           74|
|         104|         0053|           11|
|         104|         0054|            8|
|         104|         0055|           17|
|         104|         0056|           54|
|         104|         0057|           45|
|         104|         0060|            4|
|         104|         0070|            4|
|         104|         0080|            3|
|         104|         0084|            1|
+----------

## 3. What are the hardest classes?

Let's see if we can figure out which classes are the hardest by seeing how many students failed. Note that you will first need to aggregate the grades table by the course uuid to include all sections. Show the name of the course as well that you will need to get from the course_offering table.

In [93]:
from pyspark.sql import functions as F

fails_by_course = dataframes['grade_distributions'].groupBy('course_offering_uuid').agg(
    F.sum('f_count').alias('fail_count')
)

fails_with_name = fails_by_course.join(
    dataframes['course_offerings'],
    fails_by_course['course_offering_uuid'] == dataframes['course_offerings']['uuid'],
    'left'
)

fails_with_name.select('name', 'fail_count') \
    .orderBy(F.desc('fail_count')) \
    .show(20)



+--------------------+----------+
|                name|fail_count|
+--------------------+----------+
|Calc--Functns of ...|        72|
|      Animal Biology|        70|
|Calculus&Analytic...|        67|
|Calculus&Analytic...|        64|
|Calculus&Analytic...|        63|
|Calculus&Analytic...|        59|
|Calculus&Analytic...|        58|
|Calculus&Analytic...|        57|
|      Animal Biology|        56|
|      Animal Biology|        54|
|      Animal Biology|        53|
|Calculus&Analytic...|        53|
|      Animal Biology|        52|
|Calculus&Analytic...|        52|
|Intro Organic Che...|        52|
|Intro Organic Che...|        51|
|            Calculus|        50|
|Intro Organic Che...|        49|
|Intro Organic Che...|        49|
|Calculus&Analytic...|        49|
+--------------------+----------+
only showing top 20 rows


## Challenge Question: Automating data entry errors

We see in the dataframe below that there are several typos of various animal names. If this was a large database of several millions of records, correcting these errors would be way too labor intensive. How can we automate correcting these errors?

*Hint: Leven...*

In [94]:
values = [('Monkey',10),('Monkay',36),('Mnky',123), \
          ('Elephant',48),('Elefant',16),('Ellafant',1), \
          ('Hippopotamus',48),('Hipopotamus',16),('Hippo',1)]
zoo = spark.createDataFrame(values,['Animal','age'])
zoo.show()

+------------+---+
|      Animal|age|
+------------+---+
|      Monkey| 10|
|      Monkay| 36|
|        Mnky|123|
|    Elephant| 48|
|     Elefant| 16|
|    Ellafant|  1|
|Hippopotamus| 48|
| Hipopotamus| 16|
|       Hippo|  1|
+------------+---+



In [95]:
from pyspark.sql import functions as F


canonical_animals = ['Monkey', 'Elephant', 'Hippopotamus']

from pyspark.sql.types import StringType
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()
animal_df = spark.createDataFrame([(animal,) for animal in canonical_animals], ['canonical'])


crossed = zoo.crossJoin(animal_df)


crossed = crossed.withColumn('lev_dist', F.levenshtein(F.col('Animal'), F.col('canonical')))


from pyspark.sql.window import Window

window = Window.partitionBy('Animal').orderBy('lev_dist')


corrected = crossed.withColumn('rank', F.row_number().over(window)) \
    .filter(F.col('rank') == 1) \
    .select(F.col('Animal'), F.col('age'), F.col('canonical').alias('corrected_name'))

corrected.show()


[Stage 256:================================================>   (239 + 16) / 256]

+------------+---+--------------+
|      Animal|age|corrected_name|
+------------+---+--------------+
|     Elefant| 16|      Elephant|
|    Elephant| 48|      Elephant|
|    Ellafant|  1|      Elephant|
| Hipopotamus| 16|  Hippopotamus|
|       Hippo|  1|        Monkey|
|Hippopotamus| 48|  Hippopotamus|
|        Mnky|123|        Monkey|
|      Monkay| 36|        Monkey|
|      Monkey| 10|        Monkey|
+------------+---+--------------+

